# Qwen3.5-9B — LoRA Fine-Tuning a Full Precision para Verilog

**Plataforma:** Kaggle (2× T4, 30 GB VRAM, gratis)  
**Modelo base:** `Qwen/Qwen3.5-9B`  
**Técnica:** LoRA en float16 (sin cuantización — full precision)  
**Especialidad:** AXI, UART, SPI, I2C + creación/modificación de archivos `.v`/`.sv`  
**Export:** GGUF Q8_0 para LM Studio

> **Por qué full precision es posible con 9B:**  
> 9B × 2 bytes (float16) = ~18 GB de pesos + ~4 GB activaciones/LoRA = ~22 GB total.  
> 2× T4 = 30 GB → cabe sin cuantización.  
> Sin cuantización = gradientes más precisos = mejor calidad de fine-tuning.

### Antes de empezar
1. `Settings → Accelerator → GPU T4 x2`
2. Activar **Internet** en Settings
3. Correr las celdas en orden

In [ ]:
# ── Celda 1: Verificar GPUs ───────────────────────────────────────────────────
import torch, shutil, subprocess

# nvidia-smi puede no estar en PATH hasta que se instalen drivers
smi = shutil.which('nvidia-smi') or '/usr/bin/nvidia-smi'
try:
    result = subprocess.run([smi, '--query-gpu=index,name,memory.total',
                             '--format=csv,noheader'],
                            capture_output=True, text=True, check=True)
    print('GPUs disponibles:')
    print(result.stdout)
except Exception:
    print('nvidia-smi no encontrado — verificando con torch...')

n_gpus = torch.cuda.device_count()
if n_gpus == 0:
    print('❌ No se detectaron GPUs.')
    print('   Ve a Settings → Accelerator → GPU T4 x2 y reinicia el kernel.')
else:
    total_vram = sum(
        torch.cuda.get_device_properties(i).total_memory for i in range(n_gpus)
    ) / 1e9
    for i in range(n_gpus):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')
    print(f'\nTotal GPUs : {n_gpus}')
    print(f'VRAM total : {total_vram:.1f} GB')
    if total_vram < 25:
        print('⚠️  Menos de 25 GB — asegúrate de tener T4 x2 en Settings → Accelerator')

In [ ]:
# ── Celda 2: Instalar dependencias ────────────────────────────────────────────
# ⚠️  IMPORTANTE: después de correr esta celda, haz Kernel → Restart
#                 y luego corre desde la Celda 1 otra vez.
import subprocess

subprocess.run(['pip', 'install', '-q',
    'torch==2.6.0', 'torchvision==0.21.0',
    '--index-url', 'https://download.pytorch.org/whl/cu124',
])

# Instalar todo primero — trl/peft/accelerate traen su propia versión de transformers
subprocess.run(['pip', 'install', '-q',
    'trl', 'datasets', 'accelerate', 'peft',
    'tqdm', 'gitpython', 'huggingface_hub',
    'qwen-vl-utils', 'Pillow',
])

subprocess.run(['apt-get', 'install', '-qq', 'iverilog'])

# Transformers de git HEAD AL FINAL — sobreescribe cualquier versión que trl haya instalado
# Qwen3.5 requiere HEAD; además AutoModelForVision2Seq necesita versión reciente
subprocess.run(['pip', 'install', '-q', '--upgrade', '--force-reinstall',
    'git+https://github.com/huggingface/transformers.git',
])

# Verificar
import subprocess as sp
r = sp.run(['python', '-c',
    'from transformers import AutoModelForVision2Seq; print("OK")'],
    capture_output=True, text=True)
if 'OK' in r.stdout:
    print('✅ AutoModelForVision2Seq disponible')
else:
    print('❌ Error:', r.stderr[-300:])

print('✅ Dependencias instaladas.')
print('⚠️  Ahora haz: Kernel → Restart, y vuelve a correr desde la Celda 1.')

In [ ]:
# ── Celda 3: Clonar datos de entrenamiento ────────────────────────────────────
import subprocess
from pathlib import Path
from datasets import load_dataset

DATA = Path('/kaggle/working/data/raw')

# ── Repositorios Verilog ──────────────────────────────────────────────────────
REPOS = [
    ('verilog-axi',      'https://github.com/alexforencich/verilog-axi.git'),
    ('verilog-uart',     'https://github.com/alexforencich/verilog-uart.git'),
    ('verilog-i2c',      'https://github.com/alexforencich/verilog-i2c.git'),
    ('verilog-ethernet', 'https://github.com/alexforencich/verilog-ethernet.git'),
    ('wb2axip',          'https://github.com/ZipCPU/wb2axip.git'),
    ('spi-master',       'https://github.com/nandland/spi-master.git'),
    ('zipcpu',           'https://github.com/ZipCPU/zipcpu.git'),
    ('basejump-stl',     'https://github.com/bespoke-silicon-group/basejump_stl.git'),
    ('verilog-eval',     'https://github.com/NVlabs/verilog-eval.git'),
    ('RTLLM',            'https://github.com/hkust-zhiyao/RTLLM.git'),
]

for name, url in REPOS:
    dest = DATA / name
    if dest.exists():
        print(f'  skip  {name}')
    else:
        dest.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', '--depth=1', url, str(dest)], capture_output=True)
        print(f'  clone {name}')

# ── HuggingFace: dataset de texto Verilog ─────────────────────────────────────
hf_dest = DATA / 'hf' / 'shailja__Verilog_GitHub'
if not hf_dest.exists():
    print('  Descargando shailja/Verilog_GitHub...')
    ds = load_dataset('shailja/Verilog_GitHub')
    hf_dest.mkdir(parents=True, exist_ok=True)
    ds.save_to_disk(str(hf_dest))
    print(f'  Guardado: {len(ds["train"])} ejemplos')
else:
    print('  skip  shailja/Verilog_GitHub')

# ── HuggingFace: esquemáticos de circuitos (multimodal) ───────────────────────
# bshada/open-schematics: 84k esquemáticos KiCad con imagen PNG embebida
# Usamos 3000 ejemplos para no saturar el disco de Kaggle
schematics_dest = DATA / 'hf' / 'bshada__open-schematics'
if not schematics_dest.exists():
    print('  Descargando bshada/open-schematics (3000 ejemplos)...')
    ds_sch = load_dataset('bshada/open-schematics', split='train[:3000]')
    schematics_dest.mkdir(parents=True, exist_ok=True)
    ds_sch.save_to_disk(str(schematics_dest))
    print(f'  Guardado: {len(ds_sch)} esquemáticos')
else:
    print('  skip  bshada/open-schematics')

# ── HuggingFace: diagramas técnicos para preservar capacidades VL ─────────────
# lmms-lab/ai2d: 3k diagramas científicos con Q&A (imágenes embebidas en parquet)
ai2d_dest = DATA / 'hf' / 'lmms-lab__ai2d'
if not ai2d_dest.exists():
    print('  Descargando lmms-lab/ai2d...')
    ds_ai2d = load_dataset('lmms-lab/ai2d', split='test')
    ai2d_dest.mkdir(parents=True, exist_ok=True)
    ds_ai2d.save_to_disk(str(ai2d_dest))
    print(f'  Guardado: {len(ds_ai2d)} ejemplos')
else:
    print('  skip  lmms-lab/ai2d')

print('\nDatos listos.')

In [ ]:
# ── Celda 4: Preparar dataset ─────────────────────────────────────────────────
import json, re, random
from pathlib import Path
from tqdm.auto import tqdm
from datasets import load_from_disk

DATA = Path('/kaggle/working/data/raw')
OUT  = Path('/kaggle/working/data/processed')
OUT.mkdir(parents=True, exist_ok=True)

SYSTEM = """You are an expert RTL designer specialised in synthesisable Verilog and SystemVerilog.
When asked to CREATE a file, output ONLY the complete file content starting with the module declaration.
When asked to MODIFY code, output the complete modified file.
Always use non-blocking assignments (<=) in clocked always blocks.
Filenames follow snake_case (e.g. axi_lite_slave.v)."""

TAG_MAP = {
    'uart':'UART serial communication', 'spi':'SPI interface',
    'i2c':'I2C interface', 'axi':'AXI bus interface',
    'axil':'AXI-Lite bus interface', 'fifo':'synchronous FIFO',
    'arb':'round-robin arbiter', 'eth':'Ethernet MAC/PHY interface',
    'wb':'Wishbone bus interface', 'fsm':'finite state machine',
}

def module_name(src):
    m = re.search(r'\bmodule\s+(\w+)', src)
    return m.group(1) if m else None

def strip_header(src):
    lines, out, in_h = src.splitlines(), [], True
    for line in lines:
        s = line.strip()
        if in_h and (s.startswith('//') or s.startswith('/*') or s == ''):
            continue
        in_h = False
        out.append(line)
    return '\n'.join(out)

def desc(path, mod):
    stem = path.stem.lower()
    for k, v in TAG_MAP.items():
        if k in stem or k in mod.lower(): return v
    return stem.replace('_', ' ')

def make_create(path, src):
    mod = module_name(src)
    if not mod: return None
    return {'messages': [
        {'role': 'system',    'content': SYSTEM},
        {'role': 'user',      'content': f'Create a file named `{path.name}` implementing a synthesisable {desc(path, mod)} module called `{mod}` in Verilog.'},
        {'role': 'assistant', 'content': src.strip()},
    ], 'task': 'create', 'file': path.name}

def make_modify(path, src):
    if 'posedge clk' not in src: return None
    mod = module_name(src)
    if not mod: return None
    modified = src.replace('posedge rst', 'negedge rst_n').replace('if (rst)', 'if (!rst_n)')
    if modified == src: return None
    return {'messages': [
        {'role': 'system',    'content': SYSTEM},
        {'role': 'user',      'content': f'File `{path.name}`: change reset to active-low (negedge rst_n). Output the complete modified file.\n\n```verilog\n{src.strip()}\n```'},
        {'role': 'assistant', 'content': modified.strip()},
    ], 'task': 'modify', 'file': path.name}

samples = []

# Repos clonados
repo_dirs = [d for d in DATA.rglob('*') if d.is_dir() and d.name not in ('verilog-eval', 'RTLLM', 'hf')]
verilog_files = [f for d in repo_dirs for f in d.rglob('*.v')] + \
                [f for d in repo_dirs for f in d.rglob('*.sv')]
print(f'Archivos Verilog: {len(verilog_files)}')

for f in tqdm(verilog_files, desc='Repos'):
    try:
        src = f.read_text(errors='ignore')
    except Exception:
        continue
    if len(src) < 100: continue
    src = strip_header(src)
    for fn in (make_create, make_modify):
        s = fn(f, src)
        if s: samples.append(s)

# HuggingFace
hf_dest = DATA / 'hf' / 'shailja__Verilog_GitHub'
if hf_dest.exists():
    ds = load_from_disk(str(hf_dest))
    split = ds['train'] if 'train' in ds else ds
    print(f'HF: {len(split)} filas raw')
    for row in tqdm(split, desc='HF dataset'):
        inst = row.get('instruction') or row.get('prompt') or ''
        out  = row.get('output') or row.get('completion') or ''
        if not inst or not out: continue
        if not any(k in out for k in ('module ', 'endmodule')): continue
        samples.append({'messages': [
            {'role': 'system',    'content': SYSTEM},
            {'role': 'user',      'content': inst.strip()},
            {'role': 'assistant', 'content': out.strip()},
        ], 'task': 'hf_instruct'})

print(f'\nTotal samples: {len(samples)}')

random.seed(42)
random.shuffle(samples)
n_valid = max(100, int(len(samples) * 0.05))
n_test  = max(100, int(len(samples) * 0.05))
splits  = {
    'test':  samples[:n_test],
    'valid': samples[n_test:n_test + n_valid],
    'train': samples[n_test + n_valid:],
}

for name, data in splits.items():
    p = OUT / f'{name}.jsonl'
    with p.open('w') as f:
        for row in data:
            f.write(json.dumps(row) + '\n')
    print(f'  {name}.jsonl: {len(data)} samples')

In [ ]:
# ── Celda 5: QLoRA Multimodal Fine-Tuning — Qwen3.5-9B VL ───────────────────
# Dataset:
#   - Verilog texto (CREATE / MODIFY / HF_INSTRUCT)  ← capacidad principal
#   - bshada/open-schematics (esquemáticos KiCad)    ← circuito → descripción
#   - lmms-lab/ai2d (diagramas técnicos)             ← preservar VL general
# Modelo:
#   - AutoModelForVision2Seq (vision encoder incluido)
#   - Vision encoder congelado → capacidades VL preservadas
#   - LoRA r=16 solo en text decoder
import gc, torch, json, os, random
from pathlib import Path
from tqdm.auto import tqdm
from torch.utils.data import Dataset as TorchDataset
from transformers import (
    AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig,
    Trainer, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_from_disk

try:
    from qwen_vl_utils import process_vision_info
    HAS_QWEN_VL_UTILS = True
except ImportError:
    HAS_QWEN_VL_UTILS = False
    print('⚠️  qwen-vl-utils no disponible — instala en celda 2')

# ── Limpieza VRAM ─────────────────────────────────────────────────────────────
for var in ['model', 'trainer', 'merged']:
    if var in list(globals()):
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()
for i in range(torch.cuda.device_count()):
    print(f"GPU {i} tras limpieza: {torch.cuda.memory_allocated(i)/1e9:.2f} GB")

WORK     = Path('/kaggle/working')
DATA     = WORK / 'data/raw'
MODEL_ID = 'Qwen/Qwen3.5-9B'
MAX_SEQ  = 4096

# ── Procesador (tokenizador + image processor) ────────────────────────────────
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.pad_token = processor.tokenizer.eos_token

# ── Helpers ───────────────────────────────────────────────────────────────────
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

def text_to_chat(s):
    return processor.tokenizer.apply_chat_template(
        s['messages'], tokenize=False, add_generation_prompt=False,
    )

def token_len(text):
    return len(processor.tokenizer.encode(text, add_special_tokens=False))

# ── 1. Datos de texto Verilog ─────────────────────────────────────────────────
train_raw = load_jsonl(WORK / 'data/processed/train.jsonl')
valid_raw = load_jsonl(WORK / 'data/processed/valid.jsonl')
print(f'Verilog raw — Train: {len(train_raw)} | Valid: {len(valid_raw)}')

train_text, valid_text = [], []
for split_raw, split_out in [(train_raw, train_text), (valid_raw, valid_text)]:
    for s in tqdm(split_raw, desc='Filtrando texto', leave=False):
        t = text_to_chat(s)
        if token_len(t) <= MAX_SEQ:
            split_out.append({'type': 'text', 'text': t})

print(f'Verilog filtrado — Train: {len(train_text)} ({100*len(train_text)/len(train_raw):.1f}%) '
      f'| Valid: {len(valid_text)} ({100*len(valid_text)/len(valid_raw):.1f}%)')

# ── 2. Esquemáticos KiCad (bshada/open-schematics) ───────────────────────────
# Tarea: imagen del esquemático → componentes + descripción del circuito
sch_train, sch_valid = [], []
sch_path = DATA / 'hf' / 'bshada__open-schematics'
if sch_path.exists():
    ds_sch = load_from_disk(str(sch_path))
    rows = list(ds_sch)
    random.shuffle(rows)
    n_valid = max(100, len(rows) // 10)
    for row in tqdm(rows, desc='Procesando schematics', leave=False):
        try:
            img  = row['image']
            name = (row.get('name') or '').strip()
            desc = (row.get('description') or '').strip()
            comp = row.get('components_used') or []
            if not img or not (name or desc):
                continue
            comp_str = ', '.join(comp[:15]) if comp else 'no especificados'
            sample = {
                'type': 'image',
                'messages': [
                    {'role': 'user', 'content': [
                        {'type': 'image', 'image': img},
                        {'type': 'text',  'text':
                            'Analyze this circuit schematic. '
                            'List the main components and describe what this circuit does.'},
                    ]},
                    {'role': 'assistant', 'content':
                        f'**Circuit:** {name}\n'
                        f'**Components:** {comp_str}\n'
                        f'**Description:** {desc}' if desc
                        else f'**Circuit:** {name}\n**Components:** {comp_str}'},
                ],
            }
            if len(sch_valid) < n_valid:
                sch_valid.append(sample)
            else:
                sch_train.append(sample)
        except Exception:
            continue
    print(f'Schematics — Train: {len(sch_train)} | Valid: {len(sch_valid)}')
else:
    print('⚠️  open-schematics no encontrado (ejecuta celda 3)')

# ── 3. Diagramas técnicos AI2D (preservar capacidad VL general) ──────────────
ai2d_train, ai2d_valid = [], []
ai2d_path = DATA / 'hf' / 'lmms-lab__ai2d'
if ai2d_path.exists():
    ds_ai2d = load_from_disk(str(ai2d_path))
    rows = list(ds_ai2d)
    random.shuffle(rows)
    n_valid = max(50, len(rows) // 10)
    for row in tqdm(rows, desc='Procesando ai2d', leave=False):
        try:
            img     = row['image']
            question = row['question']
            opts    = row.get('options') or row.get('choices') or []
            answer  = row['answer']
            if not img or not question:
                continue
            opts_str = '\n'.join(f'{chr(65+i)}) {o}' for i, o in enumerate(opts)) if opts else ''
            user_text = f'{question}\n{opts_str}'.strip()
            sample = {
                'type': 'image',
                'messages': [
                    {'role': 'user', 'content': [
                        {'type': 'image', 'image': img},
                        {'type': 'text',  'text': user_text},
                    ]},
                    {'role': 'assistant', 'content': str(answer)},
                ],
            }
            if len(ai2d_valid) < n_valid:
                ai2d_valid.append(sample)
            else:
                ai2d_train.append(sample)
        except Exception:
            continue
    print(f'AI2D — Train: {len(ai2d_train)} | Valid: {len(ai2d_valid)}')
else:
    print('⚠️  ai2d no encontrado (ejecuta celda 3)')

# ── Combinar y mezclar ────────────────────────────────────────────────────────
all_train = train_text + sch_train + ai2d_train
all_valid = valid_text + sch_valid + ai2d_valid
random.seed(42)
random.shuffle(all_train)
print(f'\nDataset final — Train: {len(all_train)} | Valid: {len(all_valid)}')
n_img = sum(1 for s in all_train if s['type'] == 'image')
print(f'  Texto: {len(all_train)-n_img} | Imagen: {n_img}')

# ── Dataset class ─────────────────────────────────────────────────────────────
class VLDataset(TorchDataset):
    def __init__(self, samples, processor, max_length):
        self.samples    = samples
        self.processor  = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        try:
            if s['type'] == 'text':
                return self._proc_text(s['text'])
            else:
                return self._proc_image(s['messages'])
        except Exception:
            return None

    def _proc_text(self, text):
        enc = self.processor.tokenizer(
            text, return_tensors='pt',
            max_length=self.max_length, truncation=False,
        )
        ids = enc['input_ids'][0]
        return {
            'input_ids':      ids,
            'attention_mask': enc['attention_mask'][0],
            'labels':         ids.clone(),
        }

    def _proc_image(self, messages):
        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False,
        )
        if HAS_QWEN_VL_UTILS:
            image_inputs, _ = process_vision_info(messages)
        else:
            image_inputs = [
                m['content'][0]['image']
                for m in messages
                if isinstance(m.get('content'), list)
                and m['content'][0].get('type') == 'image'
            ]
        inputs = self.processor(
            text=[text], images=image_inputs, return_tensors='pt',
            max_length=self.max_length, truncation=True,
        )
        ids = inputs['input_ids'][0]
        result = {
            'input_ids':      ids,
            'attention_mask': inputs['attention_mask'][0],
            'labels':         ids.clone(),
        }
        if 'pixel_values' in inputs:
            result['pixel_values'] = inputs['pixel_values']
        if 'image_grid_thw' in inputs:
            result['image_grid_thw'] = inputs['image_grid_thw']
        return result

# ── Data collator ─────────────────────────────────────────────────────────────
class MultimodalCollator:
    def __init__(self, pad_id):
        self.pad_id = pad_id

    def __call__(self, features):
        features = [f for f in features if f is not None]
        if not features:
            return {}
        max_len = max(f['input_ids'].shape[0] for f in features)
        bs = len(features)
        input_ids      = torch.full((bs, max_len), self.pad_id, dtype=torch.long)
        attention_mask = torch.zeros(bs, max_len, dtype=torch.long)
        labels         = torch.full((bs, max_len), -100,       dtype=torch.long)
        for i, f in enumerate(features):
            n = f['input_ids'].shape[0]
            input_ids[i, :n]      = f['input_ids']
            attention_mask[i, :n] = f['attention_mask']
            labels[i, :n]         = f['labels']
        batch = {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}
        pv = [f['pixel_values'] for f in features if 'pixel_values' in f]
        if pv:
            batch['pixel_values'] = torch.cat(pv, dim=0)
            thw = [f['image_grid_thw'] for f in features if 'image_grid_thw' in f]
            if thw:
                batch['image_grid_thw'] = torch.cat(thw, dim=0)
        return batch

# ── Cargar modelo VL en 4-bit NF4 ────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = 'nf4',
    bnb_4bit_compute_dtype    = torch.float16,
    bnb_4bit_use_double_quant = True,
)

print('\nCargando modelo VL completo en 4-bit NF4...')
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config = bnb_config,
    device_map          = 'auto',
)

for i in range(torch.cuda.device_count()):
    alloc = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {alloc:.1f}/{total:.1f} GB  (libre: {total-alloc:.1f} GB)")

# ── Congelar vision encoder ───────────────────────────────────────────────────
frozen_params = 0
for name, param in model.named_parameters():
    if 'visual' in name:
        param.requires_grad = False
        frozen_params += param.numel()
print(f'Vision encoder congelado: {frozen_params/1e6:.1f}M parámetros')

# ── LoRA (solo text decoder) ──────────────────────────────────────────────────
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r              = 16,
    lora_alpha     = 32,
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                      'gate_proj','up_proj','down_proj'],
    lora_dropout   = 0.05,
    bias           = 'none',
    task_type      = 'CAUSAL_LM',
)
model = get_peft_model(model, lora_config)

# Re-congelar por si LoRA inyectó algo en visual
for name, param in model.named_parameters():
    if 'visual' in name:
        param.requires_grad = False

model.print_trainable_parameters()

# ── Build datasets y collator ─────────────────────────────────────────────────
train_ds = VLDataset(all_train, processor, MAX_SEQ)
valid_ds = VLDataset(all_valid, processor, MAX_SEQ)
collator  = MultimodalCollator(processor.tokenizer.pad_token_id)

# ── Entrenamiento ─────────────────────────────────────────────────────────────
CKPT_DIR = str(WORK / 'outputs/checkpoints')

training_args = TrainingArguments(
    output_dir                  = CKPT_DIR,
    num_train_epochs            = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    gradient_checkpointing      = True,
    warmup_steps                = 50,
    learning_rate               = 1e-4,
    lr_scheduler_type           = 'cosine',
    fp16                        = True,
    bf16                        = False,
    logging_steps               = 25,
    eval_strategy               = 'steps',
    eval_steps                  = 200,
    save_strategy               = 'steps',
    save_steps                  = 200,
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    report_to                   = 'none',
    dataloader_pin_memory       = False,
    remove_unused_columns       = False,   # crítico para pixel_values
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_ds,
    eval_dataset  = valid_ds,
    data_collator = collator,
)

checkpoints = sorted(Path(CKPT_DIR).glob('checkpoint-*'), key=os.path.getmtime)
resume_from = str(checkpoints[-1]) if checkpoints else None
if resume_from:
    print(f'Reanudando desde: {resume_from}')

print('Iniciando entrenamiento multimodal...')
trainer.train(resume_from_checkpoint=resume_from)

In [ ]:
# ── Celda 6: Benchmark con iverilog ──────────────────────────────────────────
import subprocess, tempfile, json
from pathlib import Path
from tqdm.auto import tqdm

model.eval()

def generate(instruction, max_new_tokens=768):
    messages = [
        {'role': 'system', 'content': 'You are an expert RTL designer. Output only synthesisable Verilog.'},
        {'role': 'user',   'content': instruction},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors='pt',
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs, max_new_tokens=max_new_tokens,
            temperature=0.1, do_sample=True,
        )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

def compile_check(code):
    with tempfile.NamedTemporaryFile(suffix='.v', mode='w', delete=False) as f:
        f.write(code)
        fname = f.name
    r = subprocess.run(['iverilog', '-o', '/dev/null', '-g2012', fname],
                       capture_output=True, text=True)
    return r.returncode == 0

test_samples = [json.loads(l) for l in open('/kaggle/working/data/processed/test.jsonl')][:30]
passed = 0

for s in tqdm(test_samples, desc='Benchmark'):
    user_msg  = next(m['content'] for m in s['messages'] if m['role'] == 'user')
    generated = generate(user_msg)
    if compile_check(generated):
        passed += 1

rate = passed / len(test_samples) * 100
print(f'\nCompile pass rate: {passed}/{len(test_samples)} ({rate:.1f}%)')

In [ ]:
# ── Celda 7: Exportar a GGUF Q4_K_M ──────────────────────────────────────────
# Primero fusionamos los adaptadores LoRA al modelo base, luego convertimos a GGUF.
# Q4_K_M ≈ 17 GB para 30B — cabe en el output de Kaggle (20 GB)
import subprocess
from pathlib import Path

WORK        = Path('/kaggle/working')
MERGED_DIR  = WORK / 'outputs' / 'merged'
GGUF_DIR    = WORK / 'outputs' / 'gguf'
GGUF_DIR.mkdir(parents=True, exist_ok=True)

# 1. Fusionar LoRA → modelo completo en float16
print('Fusionando adaptadores LoRA...')
merged = model.merge_and_unload()
merged.save_pretrained(str(MERGED_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(MERGED_DIR))
print(f'Modelo fusionado guardado en {MERGED_DIR}')

# 2. Instalar llama.cpp para convertir
print('Instalando llama.cpp...')
subprocess.run(['pip', 'install', '-q', 'llama-cpp-python'], check=True)

# 3. Convertir a GGUF con el script oficial de llama.cpp
print('Convirtiendo a GGUF...')
subprocess.run([
    'python', '-m', 'llama_cpp.llama_cpp.convert_hf_to_gguf',
    str(MERGED_DIR),
    '--outfile', str(GGUF_DIR / 'qwen3-coder-verilog-f16.gguf'),
    '--outtype', 'f16',
], check=True)

# 4. Quantizar a Q4_K_M
print('Quantizando a Q4_K_M...')
subprocess.run([
    'python', '-m', 'llama_cpp.llama_cpp.quantize',
    str(GGUF_DIR / 'qwen3-coder-verilog-f16.gguf'),
    str(GGUF_DIR / 'qwen3-coder-verilog-Q4_K_M.gguf'),
    'Q4_K_M',
], check=True)

for f in GGUF_DIR.glob('*.gguf'):
    print(f'{f.name}  →  {f.stat().st_size / 1e9:.2f} GB')

In [ ]:
# ── Celda 8: Subir a HuggingFace Hub ─────────────────────────────────────────
# Opción recomendada — el GGUF es grande para descargar directo desde Kaggle
# Necesitas un token HF con permisos de escritura

from huggingface_hub import HfApi
from pathlib import Path
import glob

HF_TOKEN   = 'hf_TU_TOKEN_AQUI'   # ← reemplaza con tu token
HF_REPO_ID = 'TU_USUARIO/qwen3-coder-verilog'  # ← reemplaza con tu usuario

api = HfApi(token=HF_TOKEN)

# Crear repo si no existe
api.create_repo(repo_id=HF_REPO_ID, repo_type='model', exist_ok=True)

# Subir GGUF
gguf_files = glob.glob('/kaggle/working/outputs/gguf/*.gguf')
for f in gguf_files:
    name = Path(f).name
    print(f'Subiendo {name}...')
    api.upload_file(
        path_or_fileobj = f,
        path_in_repo    = name,
        repo_id         = HF_REPO_ID,
        repo_type       = 'model',
    )
    print(f'  ✅ {name} subido a huggingface.co/{HF_REPO_ID}')

## Instalar en LM Studio

### Opción A — desde HuggingFace Hub
En LM Studio → Search → busca `TU_USUARIO/qwen3-coder-verilog` → Download

### Opción B — descarga manual
```bash
# Descargar con huggingface-cli
huggingface-cli download TU_USUARIO/qwen3-coder-verilog \
    --local-dir ~/Downloads/qwen3-coder-verilog

# Copiar a LM Studio
cp ~/Downloads/qwen3-coder-verilog/*.gguf \
   ~/Library/Application\ Support/LM\ Studio/models/
```

### Tiempos estimados en 2× T4
| Epochs | Tiempo | Notas |
|--------|--------|-------|
| 1 | ~5–7 h | cabe en una sesión Kaggle (9 h) |
| 2 | ~10–14 h | reanudar en segunda sesión |
| 3 | ~15–20 h | reanudar en tercera sesión |